In [ ]:
import pandas as pd
import json
from pydantic import BaseModel, Field
from typing import List
from enum import Enum
import boto3
import dotenv
dotenv.load_dotenv()
import os
def get_bedrock_client(region_name: str = "us-west-2"):
    return boto3.client(service_name="bedrock-runtime", region_name=region_name)

In [ ]:
bedrock_client = get_bedrock_client(region_name="us-west-2")

In [ ]:
def get_image_bytes_from_local_path(image_path: str) -> bytes:
    with open(image_path, "rb") as image_file:
        image_bytes = image_file.read()
    return image_bytes

# HFrEF

In [27]:
class MedicationCategoryHFrEF(str, Enum):
    ACE_I = "ACE-I"
    ARNI = "ARNI"
    BETA_BLOCKERS = "Beta-blockers"
    MRA = "MRA"
    SGLT2_INHIBITOR = "SGLT2 inhibitor"
    OTHER_AGENTS = "Other agents"

class HFrEFDrugDescription(BaseModel):
    generic_drug_name: str = Field(..., description="Generic name of the drug")
    starting_dose: str = Field(..., description="Starting dose amount and unit (e.g., '6.25 mg', '2.5–5 mg', '62.5 µg')")
    starting_dose_daily_intakes: int = Field(..., description="Number of daily intakes as an integer. Convert medical abbreviations: 'o.d.' -> 1, 'b.i.d.' -> 2, 't.i.d.' -> 3, 'q.i.d.' -> 4.")
    target_dose: str = Field(..., description="Target dose amount and unit (e.g., '50 mg', '10–20 mg', '250 µg')")
    target_dose_daily_intakes: int = Field(..., description="Number of daily intakes as an integer. Convert medical abbreviations: 'o.d.' -> 1, 'b.i.d.' -> 2, 't.i.d.' -> 3, 'q.i.d.' -> 4.")

class DrugClassSchema(BaseModel):
    drug_class: MedicationCategoryHFrEF = Field(..., description="Drug class")
    drugs: List[HFrEFDrugDescription] = Field(..., description="List of drugs in this class")

class HFrEFMedicationsSchema(BaseModel):
    medications: List[DrugClassSchema] = Field(..., description="List of medications grouped by class.")


class GenericDrugName(BaseModel):
    generic_drug_name: str = Field(..., description="Generic name of the drug")

class DrugClass(BaseModel):
    drug_class: MedicationCategoryHFrEF = Field(..., description="Drug class")
    drugs: List[GenericDrugName] = Field(..., description="List of drugs in this class")

class HFrEFOutputSchema(BaseModel):
    drug_classes: List[DrugClass] = Field(..., description="List of drug classes")

In [28]:
def call_bedrock_model(client, bedrock_model, prompt, response_format, description):
    conversation = [{"role": "user", "content": []}]
    for k in prompt.keys():
          if prompt[k] == "text":
               conversation[0]["content"].insert(0, {"text":k})
          elif prompt[k] == "image":
               conversation[0]["content"].insert(0, {"image":
                {'format': "png",
                 'source': {'bytes': k}}})

    response = client.converse(
        modelId=bedrock_model,
        messages=conversation,
        toolConfig={
            "tools": [
                {
                    "toolSpec": {
                        "name": "extract_medications",
                        "description": description,
                        "inputSchema": {
                            "json": response_format.model_json_schema()
                        }
                    }
                }
            ],
            "toolChoice": {"tool": {"name": "extract_medications"}}
        },
    )
    print(response["usage"])

    return response

In [29]:
model_id = "qwen.qwen3-vl-235b-a22b"

def create_prompt(file_path, image_path):
    schema_img = get_image_bytes_from_local_path(image_path)
    with open(file_path, "r", encoding="utf-8") as file:
        system_prompt = file.read()
    call_n = {system_prompt: "text",
              schema_img: "image"}
    prompt = {"\nUser: ": "text"}
    prompt.update(call_n)
    prompt["\nAI:"] = "text"
    return prompt

In [ ]:
# List of drugs
prompt_for_list = create_prompt(f"prompts/HFrEF_medication_list_prompt.txt",
                                f"images/HFrEF-medications-table.png")
answer_list = call_bedrock_model(client=bedrock_client,
                            bedrock_model=model_id,
                            prompt=prompt_for_list,
                            response_format=HFrEFOutputSchema,
                            description="Extract all drug classes and generic drug names from the table")

In [ ]:
inputs_list = [
    block["toolUse"]["input"]
    for block in answer_list["output"]["message"]["content"]
    if "toolUse" in block
]

# Salvataggio del dizionario {'drug_classes': [...]} nel file JSON
with open(f"data/HFrEF_medication_list.json", "w", encoding="utf-8") as f:
    json.dump(inputs_list[0], f, indent=2, ensure_ascii=False)

In [ ]:
# Complete table of drugs
prompt_for_table = create_prompt(f"prompts/HFrEF_medication_list_prompt.txt",
                                 f"images/HFrEF-medications-table.png")
answer_table = call_bedrock_model(client=bedrock_client,
                            bedrock_model=model_id,
                            prompt=prompt_for_table,
                            response_format=HFrEFMedicationsSchema,
                            description="Extract for all drug classes the generic drug names with starting dose, starting dose daily intakes, target dose, and target dose daily intakes from the table")

In [ ]:
with open(f"data/HFrEF_medication_list.json", "r", encoding="utf-8") as f:
    data = json.load(f)

rows = []
for drug_class in data["drug_classes"]:
    for drug in drug_class["drugs"]:
        row = {"drug_class": drug_class["drug_class"]}
        row.update(drug)
        rows.append(row)

df = pd.DataFrame(rows)
df.to_csv(f"data/HFrEF_medication_list.csv", index=False, encoding="utf-8")
print(df)

In [ ]:
inputs = [
    block["toolUse"]["input"]
    for block in answer_table["output"]["message"]["content"]
    if "toolUse" in block
]
with open(f"data/HFrEF_medication_table.json", "w", encoding="utf-8") as f:
    json.dump(inputs, f, indent=2, ensure_ascii=False)

In [ ]:
rows = []
for drug_class in inputs[0]["medications"]:
    for drug in drug_class["drugs"]:
        row = {"drug_class": drug_class["drug_class"]}
        row.update(drug)
        rows.append(row)

df = pd.DataFrame(rows)
df.to_csv(f"data/HFrEF_medication_table.csv", index=False)
print(df)